In [7]:
import sqlite3
import pandas as pd
from datetime import datetime

# connecties met sdm en dwh databases (brondatabase + dwh database afgeleid van ETL-schema's)

# bron database
sdm_conn = sqlite3.connect("BikeToDriveDatabase.db")

# data warehouse
dwh_conn = sqlite3.connect("DWH_DB.db")

In [6]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

sdm_conn.close()
dwh_conn.close()

In [10]:
# Extract (combineer beide bronnen)
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant

UNION

SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Accessoire_Verkoop_Klant
""", sdm_conn)

# Remove duplicates op business key
sdm_klant = sdm_klant.drop_duplicates(subset=['klantnr'])

# Insert met surrogate key
for _, row in sdm_klant.iterrows():
    klant_key = int(f"710{row['klantnr']}")

    dwh_conn.execute("""
    INSERT INTO Klant (klant_key, klantnr, naam, woonplaats, adres, geslacht, geboortedatum)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        klant_key,
        row['klantnr'],
        row['naam'],
        row['woonplaats'],
        row['adres'],
        row['geslacht'],
        row['geboortedatum']
    ))

dwh_conn.commit()

In [9]:
#verwijder data uit een tabel als je opnieuw wilt proberen
dwh_conn.execute("DELETE FROM Klant")
dwh_conn.commit()

In [ ]:
sdm_monteur = pd.read_sql("""
SELECT monteurnr, naam, woonplaats, uurloon
FROM Accessoire_Verkoop_Monteur

UNION

SELECT monteurnr, naam, woonplaats, uurloon
FROM Fiets_Verkoop_Monteur
""", sdm_conn)

sdm_monteur = sdm_monteur.drop_duplicates(subset=['monteurnr'])

for _, row in sdm_monteur.iterrows():
    monteur_key = int(f"710{row['monteur_key']}")

    dwh_conn.execute("""
    INSERT INTO Monteur (monteur_key, naam, woonplaats, uurloon)
    VALUES (?, ?, ?, ?)
    """, (
        monteur_key,
        row['naam'],
        row['woonplaats'],
        row['uurloon']
    ))

dwh_conn.commit()

In [ ]:
sdm_filiaal = pd.read_sql("""
SELECT filiaalnr, naam, adres, provincie
FROM Accessoire_Verkoop_Filiaal

UNION

SELECT filiaalnr, naam, adres, provincie
FROM Fiets_Verkoop_Filiaal
""", sdm_conn)

sdm_filiaal = sdm_filiaal.drop_duplicates(subset=['filiaalnr'])

for _, row in sdm_filiaal.iterrows():
    filiaal_key = int(f"710{row['filiaalnr']}")
                      
    dwh_conn.execute("""
    INSERT INTO Filiaal (filiaal_key, filiaalnr, naam, adres, provincie)
    VALUES (?, ?, ?, ?, ?)
    """, (
        filiaal_key,
        row['filiaalnr'],
        row['naam'],
        row['adres'],
        row['provincie']
    ))

dwh_conn.commit()

In [7]:
# Extract unieke datums uit SDM
sdm_datum = pd.read_sql("""
SELECT datum FROM Fiets_Verkoop
UNION
SELECT datum FROM Accessoire_Verkoop
""", sdm_conn)

# verwijder duplicates
sdm_datum = sdm_datum.drop_duplicates()

# converteer naar datetime
sdm_datum['datum'] = pd.to_datetime(sdm_datum['datum'])

# maak kolommen
sdm_datum['dag'] = sdm_datum['datum'].dt.day
sdm_datum['maand'] = sdm_datum['datum'].dt.month
sdm_datum['jaar'] = sdm_datum['datum'].dt.year
sdm_datum['kwartaal'] = sdm_datum['datum'].dt.quarter

#  maak datum_key = YYYYMMDD
sdm_datum['datum_key'] = sdm_datum['datum'].dt.strftime('%Y%m%d').astype(int)

# insert
for _, row in sdm_datum.iterrows():
    dwh_conn.execute("""
    INSERT INTO Datum (datum_key, dag, maand, kwartaal, jaar)
    VALUES (?, ?, ?, ?, ?)
    """, (
        row['datum_key'],
        row['dag'],
        row['maand'],
        row['kwartaal'],
        row['jaar']
    ))

dwh_conn.commit()

In [ ]:
sdm_lev = pd.read_sql("""
SELECT leveranciernr, naam, adres, woonplaats
FROM Accessoire_Verkoop_Leverancier

UNION

SELECT fabrikantnr AS leveranciernr, naam, adres, plaats AS woonplaats
FROM Fiets_Verkoop_Fabrikant
""", sdm_conn)

sdm_lev = sdm_lev.drop_duplicates(subset=['leveranciernr'])

for _, row in sdm_lev.iterrows():
    lev_key = int(f"710{row['leveranciernr']}")

    dwh_conn.execute("""
    INSERT INTO Leverancier (lev_key, leveranciernr, naam, adres, woonplaats)
    VALUES (?, ?, ?, ?, ?)
    """, (
        lev_key,
        row['leveranciernr'],
        row['naam'],
        row['adres'],
        row['woonplaats']
    ))

dwh_conn.commit()

In [ ]:
sdm_product = pd.read_sql("""
SELECT accessoirenr AS productnr, 'accessoire' AS product_type,
       soort, naam AS merk, NULL AS type, NULL AS kleur, leverancier AS fabrikant
FROM Accessoire_Verkoop_Accessoire

UNION

SELECT fietsnr AS productnr, 'fiets' AS product_type,
       soort, merk, type, kleur, fabrikant
FROM Fiets_Verkoop_Fiets
""", sdm_conn)

sdm_product = sdm_product.drop_duplicates(subset=['productnr'])

for i, row in sdm_product.iterrows():
    product_key = int(f"710{row['productnr']}")

    dwh_conn.execute("""
    INSERT INTO Product (product_key, productnr, product_type, soort, merk, type, kleur, fabrikant)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        product_key,
        row['productnr'],
        row['product_type'],
        row['soort'],
        row['merk'],
        row['type'],
        row['kleur'],
        row['fabrikant']
    ))

dwh_conn.commit()

In [ ]:
# Inkoop

# Extract bronnen combineren
sdm_inkoop = pd.read_sql("""
SELECT 
    i.inkoopnr, 
    i.inkoopmaand, 
    i.inkoopjaar, 
    i.aantal, 
    i.accessoire AS productnr,
    'A' AS product_type,
    a.inkoopprijs
FROM Accessoire_Inkoop i
JOIN Accessoire a 
    ON i.accessoire = a.accessoire_nr

UNION

SELECT 
    i.inkoopnr, 
    i.inkoopmaand, 
    i.inkoopjaar, 
    i.aantal, 
    i.fiets AS productnr,
    'F' AS product_type,
    f.inkoopprijs
FROM Fiets_Inkoop i
JOIN Fiets f 
    ON i.fiets = f.fiets_nr
""", sdm_conn)

# duplicanten verwijderen (business keys) 
# P.S. idk of dit wel slim is maar idk voor nu is t wel oke ig?
sdm_inkoop = sdm_inkoop.drop_duplicates(subset=['inkoopnr'])


# lookup tabellen laden
product_dim = pd.read_sql("""
SELECT product_key, productnr, product_type
FROM Product
""", dwh_conn)

periode_dim = pd.read_sql("""
SELECT periode_key, maand, jaar
FROM Periode
""", dwh_conn)


# merges
sdm_inkoop = sdm_inkoop.merge(
    product_dim,
    on=['productnr', 'product_type'],
    how='left'
)

sdm_inkoop = sdm_inkoop.merge(
    periode_dim,
    left_on=['inkoopmaand', 'inkoopjaar'],
    right_on=['maand', 'jaar'],
    how='left'
)


# insert
for _, row in sdm_inkoop.iterrows():

    dwh_conn.execute("""
    INSERT INTO Inkoop (product_key, periode_key, aantal, inkoopprijs)
    VALUES (?, ?, ?, ?)
    """, (
       row['product_key'],
       row['periode_key'],
       row['aantal'],
       row['inkoopprijs']
    ))

dwh_conn.commit()

DatabaseError: Execution failed on sql '
SELECT 
    i.inkoopnr, 
    i.inkoopmaand, 
    i.inkoopjaar, 
    i.aantal, 
    i.accessoire AS productnr,
    'A' AS product_type,
    a.inkoopprijs
FROM Accessoire_Inkoop i
JOIN Accessoire a 
    ON i.accessoire = a.accessoire_nr

UNION

SELECT 
    i.inkoopnr, 
    i.inkoopmaand, 
    i.inkoopjaar, 
    i.aantal, 
    i.fiets AS productnr,
    'F' AS product_type,
    f.inkoopprijs
FROM Fiets_Inkoop i
JOIN Fiets f 
    ON i.fiets = f.fiets_nr
': no such table: Fiets

In [ ]:
# Verkoop

# Extract (combineer beide bronnen)
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant

UNION

SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Accessoire_Verkoop_Klant
""", sdm_conn)

# Remove duplicates op business key
sdm_klant = sdm_klant.drop_duplicates(subset=['klantnr'])

# Insert met surrogate key
for _, row in sdm_klant.iterrows():
    klant_key = int(f"710{row['klantnr']}")

    dwh_conn.execute("""
    INSERT INTO Klant (klant_key, klantnr, naam, woonplaats, adres, geslacht, geboortedatum)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        klant_key,
        row['klantnr'],
        row['naam'],
        row['woonplaats'],
        row['adres'],
        row['geslacht'],
        row['geboortedatum']
    ))

dwh_conn.commit()

In [ ]:
# Onderhoud

# Extract (combineer beide bronnen)
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant

UNION

SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Accessoire_Verkoop_Klant
""", sdm_conn)

# Remove duplicates op business key
sdm_klant = sdm_klant.drop_duplicates(subset=['klantnr'])

# Insert met surrogate key
for _, row in sdm_klant.iterrows():
    klant_key = int(f"710{row['klantnr']}")

    dwh_conn.execute("""
    INSERT INTO Klant (klant_key, klantnr, naam, woonplaats, adres, geslacht, geboortedatum)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        klant_key,
        row['klantnr'],
        row['naam'],
        row['woonplaats'],
        row['adres'],
        row['geslacht'],
        row['geboortedatum']
    ))

dwh_conn.commit()

In [10]:
# Extract: SDM-data ophalen
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant AND Accessoire_Verkoop_Klant
""", sdm_conn)

# DWH-data ophalen
dwh_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Klant
""", dwh_conn)

# merge de twee tabellen
merged = sdm_klant.merge(dwh_klant, on="klantnr", how="left", suffixes=('_sdm', '_dwh'))

# pak alle rijen die nog niet in het DWH staan (nieuwe klanten)
new_rows = merged[merged['naam_dwh'].isna()]

for _, row in new_rows.iterrows():
    dwh_conn.execute("""
    INSERT INTO Klant (klantnr, naam, woonplaats, adres, geslacht, geboortedatum)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row['klantnr'],
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm']
    ))

# scd_type1
changed_rows = merged[
    (merged['naam_dwh'].notna()) &
    (
        (merged['naam_sdm'] != merged['naam_dwh']) |
        (merged['woonplaats_sdm'] != merged['woonplaats_dwh']) |
        (merged['adres_sdm'] != merged['adres_dwh']) |
        (merged['geslacht_sdm'] != merged['geslacht_dwh']) |
        (merged['geboortedatum_sdm'] != merged['geboortedatum_dwh'])
    )
]

for _, row in changed_rows.iterrows():
    dwh_conn.execute("""
    UPDATE Klant
    SET naam = ?, woonplaats = ?, adres = ?, geslacht = ?, geboortedatum = ?
    WHERE klantnr = ?
    """, (
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm'],
        row['klantnr']
    ))

dwh_conn.commit()

DatabaseError: Execution failed on sql '
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant AND Accessoire_Verkoop_Klant
': near "AND": syntax error

In [7]:
for index, row in changed_rows.iterrows():
    dwh_conn.execute("""
    UPDATE Klant
    SET naam = ?, woonplaats = ?, adres = ?, geslacht = ?, geboortedatum = ?
    WHERE klantnr = ?
    """, (
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm'],
        row['klantnr']
    ))

dwh_conn.commit()